# 实验3.2 昇腾香橙派 GPIO 驱动与 SPI 回环检测实验

**副标题**：基于 Orange Pi AI Pro 的 GPIO 驱动程序与 SPI 回环检测实践

> **运行环境说明**：本实验的**全部驱动开发与硬件操作均在香橙派 Orange Pi AI Pro 开发板上完成**。云平台中的本 Notebook（`lab3.2_cann_orange_pi_driver.ipynb`）**不需要在线运行代码**，其作用是：
> 1. 描述清楚主要代码的设计思路与关键逻辑；
> 2. 指明完整代码的存放位置（`code/` 目录）；
> 3. 说明在开发板上的编译与运行步骤；
> 4. 配合 `images/` 目录中的图片，使描述过程更加清晰。
>
> 参考文档：《实验3.1_嵌入式驱动开发实验手册》、《Orange Pi AI Pro 昇腾用户手册 v1.2》

## 一、实验目的

本实验基于**香橙派 Orange Pi AI Pro 开发板**（搭载昇腾 AI 处理器），聚焦 **GPIO 驱动程序**与 **SPI 回环检测程序**两大核心任务。通过本实验，学生将掌握：

- 使用系统自带 `spidev_test` 完成 SPI0 接口回环自检，验证收发信道完好；
- 安装配置 wiringOP 驱动库，通过 `gpio readall` 掌握 40Pin 引脚复用状态；
- 编写 `spi_loopback.c`（基于 wiringOP 的 SPI 回环检测）与 `gpio_toggle.c`（GPIO 驱动程序），以 10µs 周期翻转电平并用示波器实测信号质量；
- 理解“一套底层机制、多种上层应用”的嵌入式驱动设计思想（同一套 GPIO 翻转既可做 SPI 时钟，也可降速做 LED 闪烁）；
- 掌握“分层验证、逐层推进”的嵌入式调试方法论。

## 二、实验环境

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">环境组成</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>开发平台</strong></td>
<td style="text-align: left;">香橙派 Orange Pi AI Pro（搭载昇腾 AI 处理器）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>GPIO 驱动库</strong></td>
<td style="text-align: left;">wiringOP / wiringOP-Python（系统 <code>/usr/src</code> 自带）、libgpiod（内核标准接口）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>调试工具</strong></td>
<td style="text-align: left;">spidev_test、gpio readall、示波器、杜邦线</td>
</tr>
<tr>
<td style="text-align: left;"><strong>参考文档</strong></td>
<td style="text-align: left;">《Orange Pi AI Pro 用户手册》（深圳迅龙软件）、《实验3.1 嵌入式驱动开发实验手册》</td>
</tr>
</table>

> 开发板具有 40Pin GPIO 排针，支持 SPI/I2C/UART/PWM 等外设接口。wiringOP 是香橙派官方对标树莓派 wiringPi 构建的 C 库，将底层寄存器指令封装为易用的库函数，覆盖 GPIO 读写、SPI、I2C、PWM 等操作。


## 三、硬件接口说明：40Pin 排针与 SPI0 引脚

开发板的 40 pin 接口引脚顺序与功能定义如下图与下表所示（图片来源于《Orange Pi AI Pro 昇腾用户手册 v1.2》）。

![40Pin 接口引脚顺序](images/opi_164.png)

<div align="center">图3-1　40Pin 接口引脚顺序</div>

![40Pin 接口引脚功能表](images/opi_165.png)

<div align="center">图3-2　40Pin 接口引脚功能表</div>

### 40Pin 使用注意事项

1. 40 pin 接口中共有 26 个 GPIO 口，但 **8 号和 10 号引脚默认用于调试串口**，请不要设置为 GPIO 等功能；
2. 所有 GPIO 口的电压均为 **3.3V**；
3. 27 号和 28 号引脚只有 I2C 功能，电压默认为 1.8V。

### SPI0 关键引脚对照表

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">功能</th>
<th style="text-align: left;">物理引脚</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">MOSI (SDO)</td>
<td style="text-align: left;">19</td>
<td style="text-align: left;">主出从入，回环测试时与引脚 21 短接</td>
</tr>
<tr>
<td style="text-align: left;">MISO (SDI)</td>
<td style="text-align: left;">21</td>
<td style="text-align: left;">主入从出，回环测试时与引脚 19 短接</td>
</tr>
<tr>
<td style="text-align: left;">SCLK</td>
<td style="text-align: left;">23</td>
<td style="text-align: left;">SPI 时钟</td>
</tr>
<tr>
<td style="text-align: left;">CS</td>
<td style="text-align: left;">24</td>
<td style="text-align: left;">片选（低电平有效）</td>
</tr>
</table>

> 40 pin 接口中的 SPI 总线为 SPI0，Linux 系统默认已配置为 SPI 功能，可直接使用。测试前请先确保 `/dev` 下存在 `spidev0.0` 设备节点。


## 四、实验架构与任务总览（前因后果）

### 4.1 为什么任务顺序是“信道验证 → 装库 → 写程序”？

嵌入式驱动开发有一条铁律——**分层验证、逐层推进**：先把问题域压缩到最小、用最少依赖确认底层信道完好，再逐层向上加依赖。如果一上来就装库、写程序、接外设，一旦通讯失败，你根本分不清是**信道坏了、库没装好、还是程序写错**，排查成本成倍增加。

因此本实验的三个任务存在严格的**依赖关系**：

```
任务一  SPI0 信道验证（仅用系统工具 spidev_test，零依赖）
  │   ✅ 确认 SPI0 收发信道物理完好
  ▼
任务二  配置 wiringOP 驱动库（一次性安装库，让 -lwiringPi 可用）
  │   ✅ 确认 gpio readall 正常、库函数可调用
  ▼
任务三  基于 wiringOP 的驱动程序（spi_loopback.c + gpio_toggle.c）
        ✅ 编译运行自写的 C 程序，完成回环检测与 GPIO 翻转测试
```

### 4.2 关键澄清：任务一的回环检测需不需要 wiringOP？

**任务一只用系统自带的 `spidev_test` 工具，不需要 wiringOP。** `spidev_test` 是系统已经编译好的程序，它直接通过内核的 spidev 字符设备 `/dev/spidev0.0` 收发数据，跟 wiringOP 没有任何关系。所以任务一可以、也必须放在最前面——用零依赖的方式先确认 SPI0 物理信道是通的。

而本实验要交付的 `spi_loopback.c`（自编写回环检测程序）调用的是 wiringOP 的 `wiringPiSPISetup()` / `wiringPiSPIDataRW()`，链接 `-lwiringPi`，**它依赖 wiringOP**，所以它属于任务三，必须排在任务二之后。

> ⚠️ **不要把“用 spidev_test 验信道”和“用 spi_loopback.c 做回环”混为一谈**：前者是零依赖的系统工具（任务一），后者是依赖 wiringOP 的自写程序（任务三）。二者目标都是验证 SPI0 信道，但所处层级不同。

### 4.3 任务总览表

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">任务</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">依赖</th>
<th style="text-align: left;">核心工具 / 代码文件</th>
</tr>
<tr>
<td style="text-align: left;">任务一</td>
<td style="text-align: left;">SPI0 信道回环验证</td>
<td style="text-align: left;">无（系统工具）</td>
<td style="text-align: left;"><code>spidev_test</code></td>
</tr>
<tr>
<td style="text-align: left;">任务二</td>
<td style="text-align: left;">安装配置 wiringOP 驱动库</td>
<td style="text-align: left;">任务一通过</td>
<td style="text-align: left;"><code>./build</code>、<code>gpio readall</code></td>
</tr>
<tr>
<td style="text-align: left;">任务三</td>
<td style="text-align: left;">基于 wiringOP 的驱动程序</td>
<td style="text-align: left;">任务二通过</td>
<td style="text-align: left;"><code>code/spi_loopback.c</code>、<code>code/gpio_toggle.c</code></td>
</tr>
</table>

> **代码存放位置**：所有完整代码均放在 `code/` 目录下，本 Notebook 中仅展示关键片段与说明，完整源码请查看对应文件。


## 五、任务一：SPI0 信道验证（系统工具，无需 wiringOP）

### 5.1 原理

在接入任何外设、装任何库之前，首先要确认 SPI 接口本身是否完好。做法是**回环（Loopback）测试**：将同一 SPI 接口的 MOSI 与 MISO 短接，主机发出的数据会被原样收回，以此验证控制器、引脚复用与设备节点均工作正常。

### 5.2 操作步骤

本实验采用 SPI0 通讯口，**直接调用系统编译后自带的 `spidev_test` 测试程序即可，无需自行编写代码、无需 wiringOP**：

```bash
# 步骤1：确认 SPI0 设备节点存在
ls -l /dev/spidev0.0

# 步骤2：未短接时测试
sudo spidev_test -D /dev/spidev0.0

# 步骤3：用杜邦线短接 MOSI(物理引脚19) 与 MISO(物理引脚21)

# 步骤4：已短接时再测试
sudo spidev_test -D /dev/spidev0.0
```

![调用系统自带的 SPI 测试程序](images/exp31_02.png)

<div align="center">图5-1　调用系统自带的 SPI 测试程序</div>

### 5.3 预期结果对比

**未短接时**：可以看到发送与接收的数据不一致——因为 MISO 引脚悬空，读到的是无意义的电平状态：

![未短接时数据不一致](images/exp31_03.png)

<div align="center">图5-2　未短接时发送与接收数据不一致</div>

**短接后**：用杜邦线将 SPI0 的 MOSI（物理引脚 19）与 MISO（物理引脚 21）短接之后再次执行，发送与接收的数据完全一致，表明 SPI 收发信道畅通：

![短接后数据一致](images/exp31_04.png)

<div align="center">图5-3　短接 MOSI/MISO 后收发数据一致，信道验证通过</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">状态</th>
<th style="text-align: left;">发送数据</th>
<th style="text-align: left;">接收数据</th>
<th style="text-align: left;">结论</th>
</tr>
<tr>
<td style="text-align: left;">未短接</td>
<td style="text-align: left;"><code>0xAA</code> / <code>0x55</code></td>
<td style="text-align: left;">不一致（随机值）</td>
<td style="text-align: left;">MISO 悬空，属正常现象</td>
</tr>
<tr>
<td style="text-align: left;">已短接</td>
<td style="text-align: left;"><code>0xAA</code> / <code>0x55</code></td>
<td style="text-align: left;">完全一致</td>
<td style="text-align: left;">✅ 信道验证通过</td>
</tr>
</table>

> **前因后果小结**：任务一用零依赖的 `spidev_test` 把问题域压到最小——只验证“SPI0 控制器 + 引脚复用 + 设备节点”这三层是否贯通。通过之后，才有意义去装用户态驱动库（任务二）和写应用程序（任务三）。如果这一步就失败，后面装库写程序都是白费。


## 六、任务二：配置 wiringOP 驱动库

### 6.1 “配置 wiringOP”到底是什么意思？

这里要澄清一个容易混淆的概念：**“配置 wiringOP”不是给某个具体引脚做配置，而是把 wiringOP 这个库一次性安装到系统里**，让 `gpio` 命令行工具和 `-lwiringPi` 链接库都可用。装好之后，**所有 GPIO 引脚（包括后面要用到的引脚 40 / GPIO7_05）都可以用了**，不需要再为每个引脚单独“装”一次库。

单个引脚的“配置”指的是**设置它的方向（输入/输出）**，这一步既可以用命令行 `gpio mode` 做，也可以在 C 程序里用 `pinMode()` 做（见任务三）。本任务只负责装库。

### 6.2 安装步骤

**步骤 1：检查系统标识文件**

wiringOP 的编译脚本依赖 `/etc/orangepi-release` 文件识别开发板型号：

```bash
cat /etc/orangepi-release
```

![确认系统标识](images/exp31_05.png)

<div align="center">图6-1　确认 /etc/orangepi-release 标识为 orangepiaipro</div>

**步骤 2：复制系统自带源码包（仅供参考，建议按照下面6.2.1 下载方式安装wiringOP）**

系统已自带 wiringOP 与 wiringOP-Python 两套源码包，位于 `/usr/src` 目录下，无需从网络下载：

![系统自带源码包](images/exp31_06.png)

<div align="center">图6-2　/usr/src 目录下系统自带的 wiringOP 源码包</div>

```bash
cd ~
sudo cp -r /usr/src/wiringOP ~/
sudo cp -r /usr/src/wiringOP-Python ~/
sudo chown -R HwHiAiUser:HwHiAiUser ~/wiringOP ~/wiringOP-Python
```

**步骤 3：编译并安装**

```bash
cd ~/wiringOP
./build clean      # 清理旧编译文件
./build            # 编译
sudo ./build install   # 安装到系统
```

**步骤 4：验证安装**

```bash
gpio readall
```

安装完成后执行 `gpio readall`，可得到引脚编号、wiringPi 编号、GPIO 编号与引脚模式/电平状态的对照表。出现该表即说明 wiringOP 安装成功：

![gpio readall 输出](images/exp31_07.png)

<div align="center">图6-3　gpio readall 输出的引脚复用与状态对照表</div>

> 用户手册中 `gpio readall` 的输出示例：

![gpio readall 用户手册示例](images/opi_181.png)

<div align="center">图6-4　用户手册中 gpio readall 命令输出示例</div>

### 6.2.1 从 GitHub 在线下载安装 wiringOP（通用方法）

> 以下内容来自《Orange Pi AI Pro 昇腾用户手册》3.15 节“wiringOP 的安装使用方法”。上文 6.2 的安装步骤利用系统 `/usr/src` 自带的源码包，适用于 Orange Pi AI Pro 开发板；若系统未自带源码或需获取最新版本，可按以下通用方法从 GitHub 下载安装。

**步骤 1：确认板型标识文件**

安装 wiringOP 前，请先确保 Linux 系统中存在 `/etc/orangepi-release` 这个配置文件，里面的内容为 `BOARD=orangepiaipro`：

```bash
cat /etc/orangepi-release
# 预期输出：BOARD=orangepiaipro
```

如果 Linux 中没有 `/etc/orangepi-release` 这个配置文件，可以使用下面的命令创建一个：

```bash
echo "BOARD=orangepiaipro" | sudo tee /etc/orangepi-release
```

**步骤 2：下载 wiringOP 源码**

```bash
sudo apt-get update
sudo apt-get install -y git
git clone https://github.com/orangepi-xunlong/wiringOP.git -b next
```

> ⚠️ 源码需要下载 wiringOP **next 分支**的代码，请别漏了 `-b next` 这个参数。

**步骤 3：编译并安装 wiringOP**

```bash
sudo apt-get install -y gcc make build-essential
cd wiringOP
sudo ./build clean
sudo ./build
```

**步骤 4：测试 gpio readall 命令**

```bash
gpio readall
```

执行后可得到引脚编号、wiringPi 编号、GPIO 编号与引脚模式/电平状态的对照表，出现该表即说明 wiringOP 安装成功：

![gpio readall 输出（官方方法）](images/opi_184.png)

<div align="center">图6-2-1　从 GitHub 下载安装 wiringOP 后 gpio readall 命令输出</div>

### 6.3 命令行方式控制 GPIO（手册示例：引脚7 / GPIO7_02 / wPi 2）

用户手册以 **7 号引脚（对应 GPIO7_02，wPi 序号 2）** 为例，演示如何用 `gpio` 命令行工具设置 GPIO 方向与高低电平。**请注意这只是一个教学示例引脚，不是本实验最终要控制的引脚**：

![wiringOP 控制 GPIO 示例](images/opi_182.png)

<div align="center">图6-4　使用 wiringOP 控制 40pin GPIO 的方法（以引脚7为例）</div>

```bash
gpio mode 2 out     # 设置 wPi 2（引脚7/GPIO7_02）为输出模式
gpio write 2 0       # 输出低电平（万用表测得 0V）
gpio write 2 1       # 输出高电平（万用表测得 3.3V）
gpio read 2          # 读取引脚电平
gpio readall         # 查看所有引脚当前设置
```

![gpio readall 查看设置](images/opi_183.png)

<div align="center">图6-5　gpio readall 查看 GPIO 当前设置情况</div>

### 6.4 wiringOP 的工作原理：用户态直通硬件的开发模式

装好库、会用了 CLI，还要搞清楚 wiringOP **到底是什么、怎么和硬件对话**，才能理解本实验的驱动应用开发模式。

#### 本质：用户空间硬件操作库，而非内核驱动模块

wiringOP 是一种**用户空间硬件操作库**，而非传统意义上的内核驱动模块（`.ko`）。其核心思路是在用户态通过内存映射直接操作硬件寄存器，以此换取极高的开发效率和优秀的控制性能。这种“取巧”的方式使其成为 GPIO 控制的强大工具，在创客、教育和快速原型开发中备受欢迎。它与传统驱动层的互联，核心是通过在用户态直接操作硬件，从而**绕过了传统驱动的层层转发**。

#### 互联机制：直接与硬件对话

wiringOP 主要采用以下三种方式实现与硬件的“互联”：

**1. `/dev/mem` 内存映射（主要方式）** —— 这是 wiringOP 的核心操作方式。它通过打开 `/dev/mem` 这个特殊的设备文件，将 SoC 的 GPIO 寄存器物理地址（如 `0x01C20000`）映射到用户进程的虚拟地址空间。之后应用程序通过读写这些映射好的内存地址，直接操作硬件寄存器来控制 GPIO，从而绕过了 sysfs 或 GPIO 字符设备等标准接口。

**2. sysfs 接口（备选方式）** —— wiringOP 也提供 `wiringPiSetupSys()` 函数，通过读写 `/sys/class/gpio` 下的文件来控制 GPIO。这种方式确实会经过传统的 sysfs 驱动层，但性能不如直接内存映射。

**3. 其他标准接口** —— 对于 I2C、SPI 等复杂通信协议，wiringOP 会使用 Linux 内核提供的标准字符设备接口，如 `/dev/i2c-*` 和 `/dev/spidev*.*`。本任务的 `spi_loopback.c` 走的就是这条路径。

#### 优势：为何采用这种设计

这种“用户态库 + 直接操作硬件”的设计带来了显著优势：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><strong>开发效率极高</strong></td>
<td style="text-align: left;">封装了繁琐的寄存器操作，提供类似 Arduino 的 <code>pinMode()</code>、<code>digitalWrite()</code> 等简易 API。无需精通内核编译、设备树，也无需编写内核模块，普通 C 语言即可开发</td>
</tr>
<tr>
<td style="text-align: left;"><strong>性能与实时性</strong></td>
<td style="text-align: left;">通过 <code>/dev/mem</code> 直接操作寄存器，避免了系统调用和内核上下文切换的开销，实现极低延迟的 GPIO 控制。对需要精确时序的 1-Wire、软件 PWM、bit-banging 等应用至关重要</td>
</tr>
<tr>
<td style="text-align: left;"><strong>良好跨平台兼容性</strong></td>
<td style="text-align: left;">通过运行时检测 <code>/proc/cpuinfo</code> 自动识别硬件平台，并加载对应的寄存器基址和引脚映射表。支持超过 25 款香橙派开发板，还能兼容树莓派等平台</td>
</tr>
<tr>
<td style="text-align: left;"><strong>与驱动生态共存</strong></td>
<td style="text-align: left;">对 I2C、SPI 等复杂协议，调用内核提供的标准驱动接口，在追求简易性的同时利用了内核成熟的驱动生态</td>
</tr>
<tr>
<td style="text-align: left;"><strong>降低开发门槛</strong></td>
<td style="text-align: left;">用户空间程序崩溃通常不会导致系统宕机，调试也更简单（可用 gdb），大大降低了开发和调试门槛</td>
</tr>
</table>

#### 互联的“桥梁”：`/dev/mem`

`/dev/mem` 是 Linux 内核提供的一个特殊字符设备，可以看作是**系统物理内存的全映像**。通过它，用户空间程序能够将特定的物理内存地址（比如 GPIO 寄存器的物理地址）映射到自己的虚拟地址空间中。完成映射后，程序读写这个虚拟地址，就等同于在直接读写硬件寄存器——好比绕过了所有内核标准接口，直接拿到了操作硬件的“内部权限”。

#### 以点灯为例：技术实现流程

一个典型的 wiringOP 程序控制 LED 的流程如下：

**① 打开设备** —— 程序启动时通过 `open()` 系统调用打开 `/dev/mem`：

```c
int mem_fd = open("/dev/mem", O_RDWR | O_SYNC);
```

**② 映射内存** —— 使用 `mmap()` 将 GPIO 寄存器的物理基地址映射到用户空间虚拟地址：

```c
// 假设 GPIO_BASE 是寄存器物理基地址，映射大小为 4KB
void *gpio_map = mmap(
    NULL,                    // 让系统选择映射地址
    4096,                    // 映射大小
    PROT_READ | PROT_WRITE,  // 允许读写
    MAP_SHARED,              // 共享映射
    mem_fd,                  // /dev/mem 的文件描述符
    GPIO_BASE                // 物理地址
);
```

**③ 关闭文件** —— 一旦通过 `mmap()` 建立好内存映射，`/dev/mem` 就不再需要了，可以立即关闭：

```c
close(mem_fd);
```

**④ 操作寄存器** —— 现在 `gpio_map` 指针就指向了 GPIO 寄存器的“虚拟副本”。wiringOP 库会基于芯片手册，计算出控制 LED 的具体寄存器偏移量（如 `GPIO_OUTPUT_SET` 寄存器偏移 `0x08`），然后通过指针读写来操作：

```c
// 假设要设置 GPIO 引脚 7 为高电平
uint32_t *output_set_reg = (uint32_t*)(gpio_map + 0x08);
*output_set_reg = (1 << 7);   // 写入值，点亮 LED
```

**⑤ 清理（可选）** —— 程序结束时通过 `munmap()` 释放内存映射。

整个过程绕开了传统的 `/sys/class/gpio` 或 `/dev/gpiochip*` 这类内核驱动接口，实现了“用户态直通硬件”。

#### 小结：理解本实验的开发模式

wiringOP 与内核的“互联”并非通过常规的驱动接口，而是借用了内核提供的 `/dev/mem` 这个“后门”，将硬件寄存器的控制权直接交给用户空间程序。它的本质是一个运行在用户空间的、封装了底层寄存器操作的**内存映射库**。它避开（bypass）了传统驱动框架，以牺牲一部分安全性和标准化为代价，换取了极致的开发效率和控制性能。

> 这也解释了本实验为什么能在任务三里用几句 `wiringPiSetup()` + `pinMode()` + `digitalWrite()` 就直接控制 GPIO7_05 翻转——背后正是 wiringOP 通过 `/dev/mem` 把 GPIO 寄存器映射到用户态、由库函数替你算好偏移并直接写寄存器的结果。而 SPI 操作则走 `/dev/spidev0.0` 标准接口，两条路径在同一个库里各司其职。

> **前因后果小结**：任务二装好 wiringOP 库后，`gpio readall` 能正常输出引脚映射表，`-lwiringPi` 也能链接了。wiringOP 本质是用户态内存映射库（GPIO 走 `/dev/mem` 直通寄存器，SPI/I2C 走标准字符设备），不是内核 `.ko` 驱动。至此“库”这一层验证通过，才可以进入任务三编写并编译自写的 C 驱动程序。手册里 `gpio mode 2 out` 那个例子只是教你 CLI 怎么用，真正的引脚配置在任务三的 C 程序里用 `pinMode()` 完成。


## 七、任务三：基于 wiringOP 的驱动程序

本任务交付两个自编 C 程序，**二者都依赖任务二安装的 wiringOP 库**（链接 `-lwiringPi`），因此必须排在任务二之后：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">程序</th>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">功能</th>
</tr>
<tr>
<td style="text-align: left;">SPI 回环检测</td>
<td style="text-align: left;"><code>code/spi_loopback.c</code></td>
<td style="text-align: left;">用 wiringOP 的 SPI 接口做回环自检，自动判定 PASS/FAIL</td>
</tr>
<tr>
<td style="text-align: left;">GPIO 驱动测试</td>
<td style="text-align: left;"><code>code/gpio_toggle.c</code></td>
<td style="text-align: left;">用 wiringOP 的 GPIO 接口翻转引脚 40，示波器实测信号</td>
</tr>
</table>

### 7.1 SPI 回环检测程序 `spi_loopback.c`

#### 设计思路

任务一已用系统 `spidev_test` 验过信道，这里再用**自写的 C 程序**做一遍回环检测，目的是掌握 wiringOP 的 SPI 编程接口。核心逻辑：

1. 调用 `wiringPiSPISetup(0, 1000000)` 初始化 SPI0，速率 1MHz；
2. 准备 8 字节递增测试数据 `00 01 02 ... 07`；
3. 调用 `wiringPiSPIDataRW()` 同时发送并接收（数据原地覆盖）；
4. 用 `memcmp()` 逐字节比对收发数据，一致则输出 `[PASS]`，否则 `[FAIL]`；
5. 支持命令行参数指定测试次数，结束后输出通过率统计。

**关键代码片段**（完整代码见 `code/spi_loopback.c`）：

```c
/* 初始化 SPI0 */
int fd = wiringPiSPISetup(SPI_CHANNEL, SPI_SPEED);   // channel=0, speed=1MHz

/* 收发数据（核心函数，数据原地覆盖） */
wiringPiSPIDataRW(SPI_CHANNEL, rx_buf, BUF_SIZE);

/* 逐字节比对，判定回环是否通过 */
int success = (memcmp(tx_buf, rx_buf, BUF_SIZE) == 0);
```

#### 编译与运行

```bash
cd code
gcc spi_loopback.c -o spi_loopback -lwiringPi
# 或用 Makefile：make spi

sudo ./spi_loopback        # 持续循环测试（每秒一次）
sudo ./spi_loopback 5      # 测试 5 次后退出并输出通过率
```

#### 预期结果

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">状态</th>
<th style="text-align: left;">发送</th>
<th style="text-align: left;">接收</th>
<th style="text-align: left;">程序输出</th>
<th style="text-align: left;">结论</th>
</tr>
<tr>
<td style="text-align: left;">未短接</td>
<td style="text-align: left;"><code>00 01 02 03 04 05 06 07</code></td>
<td style="text-align: left;">随机值</td>
<td style="text-align: left;"><code>[FAIL] 数据不一致</code></td>
<td style="text-align: left;">MISO 悬空，正常</td>
</tr>
<tr>
<td style="text-align: left;">已短接</td>
<td style="text-align: left;"><code>00 01 02 03 04 05 06 07</code></td>
<td style="text-align: left;">完全一致</td>
<td style="text-align: left;"><code>[PASS] 数据一致</code></td>
<td style="text-align: left;">✅ 回环通过</td>
</tr>
</table>

> 完整预期输出对照见 `answer/spi_loopback_output.md`。


### 7.2 GPIO 驱动程序 `gpio_toggle.c`

#### 设计思路

将香橙派 AI Pro 的**物理引脚 40**设为输出，以 10µs 半周期翻转电平，用示波器实测 GPIO 翻转速率与信号质量。核心逻辑：

1. 调用 `wiringPiSetup()` 初始化 wiringOP（wPi 编号体系）；
2. 调用 `pinMode(25, OUTPUT)` 将物理引脚 40 设为输出；
3. 通过 `SCHED_FIFO` 实时调度与 `nice=-20` 提升进程优先级，减少调度抖动；
4. 主循环中以 `delayMicroseconds(10)` 交替输出高低电平，形成 10µs 半周期方波；
5. 支持命令行参数指定半周期，加大后即可降速为 LED 闪烁。

#### 引脚编号详解：手册示例的 GPIO7_02 与本程序的 GPIO7_05 是什么关系？

这是一个很容易困惑的点，讲清楚：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;"></th>
<th style="text-align: left;">手册 CLI 示例（任务二）</th>
<th style="text-align: left;">本 GPIO 程序（任务三）</th>
</tr>
<tr>
<td style="text-align: left;">物理引脚</td>
<td style="text-align: left;">7</td>
<td style="text-align: left;"><strong>40</strong></td>
</tr>
<tr>
<td style="text-align: left;">GPIO 命名</td>
<td style="text-align: left;">GPIO7_02</td>
<td style="text-align: left;"><strong>GPIO7_05</strong></td>
</tr>
<tr>
<td style="text-align: left;">wPi 编号</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><strong>25</strong></td>
</tr>
<tr>
<td style="text-align: left;">GPIO 全局编号</td>
<td style="text-align: left;">226（=7×32+2）</td>
<td style="text-align: left;"><strong>229</strong>（=7×32+5）</td>
</tr>
<tr>
<td style="text-align: left;">用途</td>
<td style="text-align: left;">教学演示 <code>gpio mode 2 out</code></td>
<td style="text-align: left;">示波器实测翻转波形</td>
</tr>
</table>

> 上述映射来自 wiringOP 库源码（开发板 `/usr/src/wiringOP/wiringPi/wiringPi.c`）中的 `pinToGpio_AIPRO[25]=229` 与 `physToGpio_AIPRO[40]=229`，229 = 7×32+5 即 GPIO7_05。手册的 7 号引脚示例（GPIO7_02 / wPi 2）**只是一个教你用 CLI 的例子**，本实验要控制的是 40 号引脚（GPIO7_05 / wPi 25），二者同属 GPIO7 组但不是同一个脚。

#### 关键澄清：GPIO7_05 没有单独“配置 wiringOP”吗？还是调用函数时就配置了？

**答案是：调用 `pinMode(25, OUTPUT)` 时就在运行时完成了配置，不需要再单独跑一次 `gpio mode` 命令。**

wiringOP 的“配置”分两层，不要混淆：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层级</th>
<th style="text-align: left;">做什么</th>
<th style="text-align: left;">在哪做</th>
<th style="text-align: left;">是否针对单个引脚</th>
</tr>
<tr>
<td style="text-align: left;"><strong>库级安装</strong></td>
<td style="text-align: left;">把 wiringOP 装进系统，让所有引脚都可用</td>
<td style="text-align: left;">任务二 <code>./build install</code>（一次性）</td>
<td style="text-align: left;">❌ 对所有引脚生效</td>
</tr>
<tr>
<td style="text-align: left;"><strong>引脚级配置</strong></td>
<td style="text-align: left;">把某个引脚设为输入/输出方向</td>
<td style="text-align: left;">任务三 C 程序里 <code>pinMode()</code></td>
<td style="text-align: left;">✅ 针对具体引脚</td>
</tr>
</table>

具体到 `gpio_toggle.c`：

```c
wiringPiSetup();          // ① 库级初始化：加载引脚映射表、映射寄存器，对整个 GPIO 子系统生效
pinMode(25, OUTPUT);       // ② 引脚级配置：把 GPIO7_05(引脚40) 设为输出，等价于命令行 gpio mode 25 out
digitalWrite(25, HIGH);    // ③ 输出电平：等价于 gpio write 25 1
```

- `wiringPiSetup()` 只需调用一次，它**不设置任何引脚的方向**，只是初始化库（建立 wPi→GPIO 的映射、打开寄存器访问）；
- `pinMode(25, OUTPUT)` 才是真正“配置 GPIO7_05”的那一步——把它设为输出方向，在程序运行时完成，**等价于命令行的 `gpio mode 25 out`**；
- 所以 GPIO7_05 不需要在任务二里额外配置，它的配置就在任务三的程序里由 `pinMode()` 一句搞定。

**关键代码片段**（完整代码见 `code/gpio_toggle.c`）：

```c
/* 物理引脚 40 = wPi 编号 25 = GPIO7_05（GPIO 全局编号 229 = 7*32+5） */
#define PIN 25

wiringPiSetup();               // 初始化 wiringOP（wPi 编号体系）
pinMode(PIN, OUTPUT);          // 将 GPIO7_05 设为输出（运行时配置）

/* 提升实时优先级，减少调度抖动 */
setpriority(PRIO_PROCESS, 0, -20);
struct sched_param param;
param.sched_priority = sched_get_priority_max(SCHED_FIFO);
sched_setscheduler(0, SCHED_FIFO, &param);

/* 主循环：10µs 周期翻转电平 */
while (1) {
    digitalWrite(PIN, HIGH);
    delayMicroseconds(10);
    digitalWrite(PIN, LOW);
    delayMicroseconds(10);
}
```

#### 编译与运行

```bash
cd code
gcc gpio_toggle.c -o gpio_toggle -lwiringPi
# 或用 Makefile：make gpio

sudo ./gpio_toggle            # 10µs 周期翻转（理论 50kHz），示波器接引脚 40
sudo ./gpio_toggle 500000     # 半周期 0.5s，LED 慢闪
```

#### 示波器验证结果

示波器探头接**物理引脚 40**，GND 接开发板任意 GND 引脚。实测波形如下：约 10µs 周期，波形规则、边沿清晰，说明 GPIO 翻转信号质量良好：

![示波器实测波形](images/exp31_09.png)

<div align="center">图7-1　示波器实测 GPIO 翻转波形（约 10µs 周期）</div>

![示波器测试场景](images/exp31_10.png)

<div align="center">图7-2　示波器连接香橙派引脚的实际测试场景</div>

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">观测项</th>
<th style="text-align: left;">预期结果</th>
</tr>
<tr>
<td style="text-align: left;">波形形状</td>
<td style="text-align: left;">规则方波，边沿陡峭清晰</td>
</tr>
<tr>
<td style="text-align: left;">电平幅度</td>
<td style="text-align: left;">0 ~ 3.3V 完整摆幅</td>
</tr>
<tr>
<td style="text-align: left;">时钟周期</td>
<td style="text-align: left;">约 20µs（半周期 10µs）</td>
</tr>
<tr>
<td style="text-align: left;">实测频率</td>
<td style="text-align: left;">约 45~48kHz（受 <code>digitalWrite</code> 耗时影响，略低于理论 50kHz）</td>
</tr>
<tr>
<td style="text-align: left;">周期稳定性</td>
<td style="text-align: left;">周期均匀一致（已提升 SCHED_FIFO 实时优先级）</td>
</tr>
</table>

> 完整预期输出对照见 `answer/gpio_toggle_output.md`。

#### 降速为 LED 闪烁（一套底层机制、多种上层应用）

```bash
sudo ./gpio_toggle 500000    # 半周期 0.5s，LED 以 1Hz 闪烁
```

在引脚 40 与 GND 之间串接 LED + 330Ω 限流电阻，可观察到 LED 以 0.5s 亮、0.5s 灭的频率闪烁。这说明**同一套 GPIO 电平翻转机制，改变速率即可从高速 SPI 时钟降速为 LED 闪烁**——这正是嵌入式驱动开发中“一套底层机制、多种上层应用”的典型体现。

> **前因后果小结**：任务三的两个程序都建立在任务二装好的 wiringOP 之上。`spi_loopback.c` 复用任务一的回环思路但改用 wiringOP 的 SPI 接口；`gpio_toggle.c` 控制 GPIO7_05（引脚 40），其引脚配置由 `pinMode()` 在运行时完成，无需额外命令行配置。至此“信道→库→程序”三层全部打通。


## 八、代码文件位置与编译总览

### 8.1 目录结构

```
Lab3_2/
├── lab3.2_cann_orange_pi_driver.ipynb   # 本实验说明 Notebook
├── code/                                 # ★ 完整代码目录
│   ├── gpio_toggle.c                     # GPIO 驱动测试程序（引脚40/GPIO7_05）
│   ├── spi_loopback.c                    # SPI0 回环检测程序（基于 wiringOP）
│   ├── Makefile                          # 编译脚本
│   └── README.md                         # 代码目录说明
├── images/                               # 实验图片目录（共 13 张，均被本 Notebook 引用）
│   ├── exp31_02~07、09、10.png (8 张)    # 来源于实验3.1手册
│   └── opi_164、165、181、182、183.png   # 来源于 OrangePi 用户手册（5 张）
└── answer/                               # 参考答案目录
    ├── README.md                         # 参考答案总览
    ├── spi_loopback_output.md            # SPI 回环预期输出
    ├── gpio_toggle_output.md             # GPIO 测试预期输出
    └── lab_report_key_points.md               # 实验报告撰写参考
```

> **说明**：本目录即本实验交付物的全部内容。`images/` 中的图片提取自《实验3.1 嵌入式驱动开发实验手册》与《Orange Pi AI Pro 昇腾用户手册 v1.2》两份文档（原文档及原始测试代码 `testcode/`、中间产物 `output/` 已不在交付目录中）。wiringOP 库源码位于开发板 `/usr/src/wiringOP`，由系统自带。

### 8.2 一键编译与运行（在香橙派开发板上）

```bash
cd code
make                        # 一键编译全部程序

sudo ./spi_loopback 5       # SPI 回环检测（短接引脚19与21后应显示 [PASS]）
sudo ./gpio_toggle          # GPIO 驱动测试（示波器接引脚40观察波形）
sudo ./gpio_toggle 500000   # 降速为 LED 闪烁

make clean                  # 清理编译产物
```

### 8.3 代码与参考来源对照

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">生成代码</th>
<th style="text-align: left;">参考来源</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>code/spi_loopback.c</code></td>
<td style="text-align: left;">实验3.1 测试代码 <code>spi_loopback.c</code></td>
<td style="text-align: left;">增加命令行参数、通过率统计与完整注释</td>
</tr>
<tr>
<td style="text-align: left;"><code>code/gpio_toggle.c</code></td>
<td style="text-align: left;">实验3.1 测试代码 <code>gpio_10us.c</code></td>
<td style="text-align: left;">增加可配置半周期、实时优先级封装与完整注释</td>
</tr>
<tr>
<td style="text-align: left;"><code>code/Makefile</code></td>
<td style="text-align: left;">新增</td>
<td style="text-align: left;">一键编译脚本</td>
</tr>
</table>


## 九、常见问题与排查方法

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题现象</th>
<th style="text-align: left;">可能原因</th>
<th style="text-align: left;">排查与解决</th>
</tr>
<tr>
<td style="text-align: left;"><code>spidev_test</code> 报错或无设备</td>
<td style="text-align: left;">SPI 设备节点未使能</td>
<td style="text-align: left;">检查 <code>/dev/spidev0.0</code> 是否存在，确认 SPI0 已在系统中使能</td>
</tr>
<tr>
<td style="text-align: left;">wiringOP 编译失败</td>
<td style="text-align: left;">板型标识不对或权限不足</td>
<td style="text-align: left;">确认 <code>/etc/orangepi-release</code> 为 <code>orangepiaipro</code>，并 <code>chown</code> 源码目录到当前用户</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio readall</code> 无输出</td>
<td style="text-align: left;">wiringOP 未正确安装</td>
<td style="text-align: left;">重新执行 <code>./build</code> 与 <code>sudo ./build install</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>spi_loopback</code> 始终 <code>[FAIL]</code></td>
<td style="text-align: left;">未短接 MOSI/MISO</td>
<td style="text-align: left;">用杜邦线短接物理引脚 19 与 21</td>
</tr>
<tr>
<td style="text-align: left;">GPIO 波形抖动严重</td>
<td style="text-align: left;">进程优先级不足</td>
<td style="text-align: left;">确认以 <code>sudo</code> 运行，程序已设置 <code>SCHED_FIFO</code> 实时调度</td>
</tr>
<tr>
<td style="text-align: left;">实测频率低于理论值</td>
<td style="text-align: left;"><code>digitalWrite</code> 函数调用耗时</td>
<td style="text-align: left;">属正常现象，约 45~48kHz；如需更高精度可改用寄存器直接操作</td>
</tr>
<tr>
<td style="text-align: left;">控制引脚 40 没反应</td>
<td style="text-align: left;">引脚方向未设为输出</td>
<td style="text-align: left;">确认程序调用了 <code>pinMode(25, OUTPUT)</code>；<code>wiringPiSetup()</code> 不会自动设方向</td>
</tr>
</table>


## 十、实验总结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">知识点</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">分层验证方法论</td>
<td style="text-align: left;">先用零依赖的 spidev_test 验信道→再装库→最后写程序，逐层推进</td>
</tr>
<tr>
<td style="text-align: left;">wiringOP 本质</td>
<td style="text-align: left;">用户态内存映射库（非内核 .ko）：GPIO 走 /dev/mem 直通寄存器，SPI/I2C 走标准字符设备</td>
</tr>
<tr>
<td style="text-align: left;">wiringOP 库级 vs 引脚级配置</td>
<td style="text-align: left;">库安装（<code>./build install</code>）对所有引脚生效；引脚方向由 <code>pinMode()</code> 在运行时逐脚配置</td>
</tr>
<tr>
<td style="text-align: left;">SPI 回环检测</td>
<td style="text-align: left;">任务一用系统工具、任务三用自写程序，两层都验证 SPI0 信道</td>
</tr>
<tr>
<td style="text-align: left;">GPIO 驱动程序</td>
<td style="text-align: left;"><code>wiringPiSetup</code> + <code>pinMode</code> + <code>digitalWrite</code> + <code>delayMicroseconds</code> 实现电平翻转</td>
</tr>
<tr>
<td style="text-align: left;">引脚编号体系</td>
<td style="text-align: left;">物理引脚40 = wPi 25 = GPIO7_05（全局229）；手册示例引脚7 = GPIO7_02 = wPi 2</td>
</tr>
<tr>
<td style="text-align: left;">实时优先级</td>
<td style="text-align: left;"><code>SCHED_FIFO</code> + <code>nice=-20</code> 减少调度抖动，稳定波形周期</td>
</tr>
<tr>
<td style="text-align: left;">一套机制多种应用</td>
<td style="text-align: left;">同一套 GPIO 翻转既可做 SPI 时钟，也可降速做 LED 闪烁</td>
</tr>
</table>

### 最终效果

- 任务一：`spidev_test` 短接后收发数据一致，SPI0 信道验证通过；
- 任务二：`gpio readall` 正常输出引脚映射表，wiringOP 安装成功；
- 任务三：`spi_loopback` 短接后输出 `[PASS]`；`gpio_toggle` 示波器实测约 10µs 周期方波，边沿清晰；`gpio_toggle 500000` 可见 LED 慢闪。

## 十一、实验完成标志

- [ ] 任务一：`spidev_test` 短接引脚 19/21 后收发数据一致
- [ ] 任务二：`gpio readall` 正常输出引脚映射表
- [ ] 任务三：`spi_loopback` 短接后显示 `[PASS]`
- [ ] 任务三：`gpio_toggle` 示波器实测 GPIO7_05（引脚40）波形规则（约 10µs 周期）
- [ ] 任务三：`gpio_toggle 500000` 可见 LED 闪烁效果

## 十二、参考资料

- 《实验3.1 嵌入式驱动开发实验手册》（GPIO 驱动与 AI 算力协同开发）
- 《Orange Pi AI Pro 昇腾用户手册 v1.2》（深圳迅龙软件）
- [wiringOP GitHub](https://github.com/orangepi-xunlong/wiringOP)
- [昇腾 CANN 文档](https://www.hiascend.com/document)

> **文件位置索引**：完整代码 → `code/`；参考答案 → `answer/`；实验图片 → `images/`。
